# 04 — Train on Colab / Kaggle (free GPU)

Tiling and normalization happen locally or here; the U-Net training needs a GPU.
Set **Runtime → Change runtime type → GPU** (Colab) or enable a GPU accelerator (Kaggle) first.

This notebook is a thin wrapper around `src/` — the same scripts run locally, just slower.


In [ ]:
# GPU check
!nvidia-smi -L || echo 'No GPU — set Runtime > Change runtime type > GPU'

In [ ]:
# Deps. Colab/Kaggle ship a CUDA build of torch already -- do NOT reinstall the
# CPU-pinned torch from requirements.txt or you'll lose the GPU.
!pip install -q segmentation-models-pytorch rasterio scipy pandas scikit-learn matplotlib folium timm

In [ ]:
# Get the code: clone your repo, or upload the src/ folder.
# !git clone https://github.com/<you>/deforestation-detection.git
# %cd deforestation-detection
import os
print('cwd:', os.getcwd())

In [ ]:
# --- Option A: synthetic smoke test (no Earth Engine, runs right now) ---
!python src/data/make_synthetic.py --out /content/synth --size 1024
!python src/data/prepare_tiles.py --image /content/synth/t1.tif --label /content/synth/label.tif --out /content/tiles --block-size 512

In [ ]:
# --- Option B: real data ---
# 1) earthengine authenticate  + export the composites/labels (see notebooks/01-02)
# 2) download the GeoTIFFs, then:
# !python src/data/prepare_tiles.py --image data/raw/s2_colniza_2020.tif --label data/raw/dw_trees_2020.tif --out /content/tiles --block-size 1024

In [ ]:
# Train: U-Net, resnet34 encoder, Dice+BCE, flips/rotations, fixed seed
!python src/train.py --data-dir /content/tiles --epochs 30 --batch-size 16 --out /content/unet.pth --metrics /content/metrics.csv --region 'Colniza, MT'

In [ ]:
# Evaluate on the held-out area (pass the NDVI baseline IoU from step 4)
!python src/evaluate.py --data-dir /content/tiles --split test --checkpoint /content/unet.pth --baseline-iou 0.0 --metrics /content/metrics.csv --region 'Colniza, MT'

In [ ]:
import pandas as pd
pd.read_csv('/content/metrics.csv')

In [ ]:
# Save the checkpoint back to Drive so it survives the runtime
# from google.colab import drive; drive.mount('/content/drive')
# !cp /content/unet.pth /content/drive/MyDrive/deforestation/unet.pth